<a href="https://colab.research.google.com/github/CesarChaMal/ollama/blob/main/ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
# Linux uninstall ollama
!rm -rf *
!echo $OLLAMA_HOST
!export OLLAMA_HOST=http://localhost:11434
import os
os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
!echo $OLLAMA_HOST
!command -v systemctl >/dev/null && ps aux | grep 'ollama' | grep -v grep | awk '{print $2}' | xargs -r kill
!rm /etc/systemd/system/ollama.service
!sudo rm $(which ollama)
!sudo rm -r /usr/share/ollama
!sudo userdel ollama

# Download and run the Ollama Linux install script
!curl https://ollama.ai/install.sh | sh

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers
#!ollama start
!nohup ollama start &
!nohup ollama serve &
!ps aux | grep ollama
!journalctl -u ollama.service


http://localhost:11434
http://localhost:11434
rm: cannot remove '/etc/systemd/system/ollama.service': No such file or directory
rm: cannot remove '/usr/share/ollama': No such file or directory
userdel: user 'ollama' does not exist
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0>>> Downloading ollama...
100  8354    0  8354    0     0  31224      0 --:--:-- --:--:-- --:--:-- 31171
############################################################################################# 100.0%
>>> Installing ollama to /usr/local/bin...
>>> Creating ollama user...
useradd: group ollama exists - if you want to add this user to that group, use -g.
>>> The Ollama API is now available at 0.0.0.0:11434.
>>> Install complete. Run "ollama" from the command line.
Hit:1 https://developer.download.nvidia.com/compute/cuda/repo

In [ ]:
!rm -rf *

!pip install aiohttp pyngrok
!pip show pyngrok
!pip install --upgrade pyngrok

import os
import asyncio
from aiohttp import ClientSession
from pyngrok import ngrok

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

async def run(cmd):
    '''
    run is a helper function to run subcommands asynchronously.
    '''
    print('>>> starting', *cmd)
    p = await asyncio.subprocess.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    async def pipe(lines):
        async for line in lines:
            print(line.strip().decode('utf-8'))

    await asyncio.gather(
        pipe(p.stdout),
        pipe(p.stderr),
    )


#await asyncio.gather(
#    run(['ollama', 'serve']),
#    run(['ngrok', 'http', '--log', 'stderr', '11434']),
#)


async def run_ollama_commands():
    # Define the commands you want to run
    commands = [
        ['ollama', 'list'],
        ['ollama', 'run', 'mistral']
    ]

    for cmd in commands:
        print('>>> Executing:', ' '.join(cmd))
        process = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE,
                                                       stderr=asyncio.subprocess.PIPE)

        # Capture the output
        stdout, stderr = await process.communicate()
        if stdout:
            print(stdout.decode())
        if stderr:
            print(stderr.decode())


# Run the ollama commands
await run_ollama_commands()
#ngrok.disconnect(public_url.public_url)


# Set up ngrok
ngrok.set_auth_token("")
public_url = ngrok.connect(11434, "http")
print("Ngrok Tunnel URL:", public_url)
!cat /tmp/ngrok.log

# Set the OLLAMA_HOST environment variable
#os.environ['OLLAMA_HOST'] = public_url.public_url
#os.environ['OLLAMAUI_HOST'] = public_url.public_url



The previous cell starts two processes, `ollama` and `ngrok`. The log output will show a line like the following which describes the external address.

```
t=2023-11-12T22:55:56+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://localhost:11434 url=https://8249-34-125-179-11.ngrok.io
```

The external address in this case is `https://8249-34-125-179-11.ngrok.io` which can be passed into `OLLAMA_HOST` to access this instance.

```bash
export OLLAMA_HOST=https://8249-34-125-179-11.ngrok.io
ollama list
ollama run mistral
```

In [ ]:
!export OLLAMA_HOST=https://8249-34-125-179-11.ngrok.io
!ollama list
!ollama run mistral